# A CGS Implementation for a 3-Storey Building Construction Sequencing

[Rizal Purnawan](https://orcid.org/0000-0001-8858-4036)<sup>1</sup>,
[Angga T. Yudhistira](https://orcid.org/0000-0002-7264-7810)<sup>2</sup>,
[Christoporus A. A. Ohmar](https://orcid.org/0009-0004-2074-3263)<sup>3</sup>

<sup>1</sup>Independent Researcher, Surabaya, Indonesia \
<sup>2</sup>Universitas Gajah Mada, Indonesia \
<sup>3</sup>Independent Researcher, Yogyakarta, Indonesia

In [1]:
# Library
from maxplus import *    # maxplus module
import numpy as np
import pandas as pd
import time
from IPython.display import Latex

In [2]:
# ALGORITHM

# Utilization Map
def rho(a, x, y, n):
    n1, n2 = n
    if a[0] == "p":
        r = (n1 *n2) % x
        q = (n1 *n2) // x
        i = int(a.replace("p", ""))
    elif a[0] == "s":
        r = n2 % y
        q = n2 // y
        i = int(a.replace("s", ""))
    else:
        raise ValueError
    return q + 1 if 0 < i <= r else q


# PG, SG and EG
def primG(x):
    Ap = [f"p{i}" for i in range(x)]
    return [
        (a, k)
        for a in Ap
        for k in range(1, rho(a, x, 0, n) + 1)
    ]

def secG(y):
    As = [f"s{i}" for i in range(y)]
    return [
        (a, k)
        for a in As
        for k in range(1, rho(a, 0, y, n) + 1)
    ]


# Lexicographic Ordering
def lamb(c, x, y, n):
    n1, n2 = n
    Ep = primG(x)
    Es = secG(y)
    a, k = c

    if c in Ep:
        i = int(a.replace("p", ""))
        kappa = i + (k - 1) *x
        return kappa + floor((kappa - 1) / n1)
    elif c in Es:
        i = int(a.replace("s", ""))
        return (n1 + 1) *(i + (k - 1) *y)
    else:
        raise ValueError


# Anti-Lexicographic Ordering
def mu(m, x, y, n):
    n1, n2 = n
    if m % (n1 + 1) > 0:
        k = floor((m - floor(m / (n1 + 1)) -1) / x) + 1
        i = m - floor(m / (n1 + 1)) - (k - 1) *x
        a = f"p{i}"
    elif m % (n1 + 1) == 0:
        k = floor((m / (n1 + 1) -1) / y) + 1
        i = int(m / (n1 + 1)) - (k - 1) *y
        a = f"s{i}"
    else:
        raise ValueError
    return (a, k)


# Generating CGS-DM
def cgsDM(G):
    x, y, n, d = G
    d1, d2, d3, d4 = d
    n1, n2 = n
    dim = (n1 + 1) *n2
    R = [[E for k in range(dim)] for k in range(dim)]
    for p in range(dim):
        for q in range(dim):
            i, j = p + 1, q + 1
            # Procedure 1:
            if i == j + 1 and i % (n1 + 1) != 1:
                R[p][q] = R[p][q] |op| (d1 |ot| 1)

            # Procedure 2 & 3:
            ai, ki = mu(i, x, y, n)   # For Proc. 3
            aj, kj = mu(j, x, y, n)   # For Proc. 3
            if i == j + (n1 + 1) or i == j + 2 *(n1 + 1) \
                    or (ai == aj and ki == kj + 1):
                R[p][q] = R[p][q] |op| oprod(d)
    return R


# First Objective Function
def psi(G):
    x, y, n, d = G
    n1, n2 = n
    dim = (n1 + 1) *n2
    Delta = oprod(d)
    R_exp = [cgsDM(G)|exp|k for k in range(dim)]
    R_ast = osum(R_exp)
    e1 = [0] + ([E] *(dim -1))
    return Delta |ot| osum(R_ast |ot| e1)


# Optimization
def optimizeCGS(search_space, runtime= False):
    start_time = time.perf_counter()
    ##
    # Step O1:
    fo_min = psi(search_space[0])
    G_prime = [search_space[0]]
    for G in search_space[1:]:
        fo = psi(G)
        if fo < fo_min:
            G_prime = [G]
            fo_min = fo
        elif fo == fo_min:
            G_prime.append(G)
    ###
    # Step O2:
    xy_min = G_prime[0][0] + G_prime[0][1]
    G_prime2 = [G_prime[0]]
    for G in G_prime[1:]:
        xy = G[0] + G[1]
        if xy < xy_min:
            G_prime2 = [G]
            xy_min = xy
        elif xy == xy_min:
            G_prime2.append(G)
    ###
    # Step O3:
    y_min = G_prime2[0][1]
    G_opt = G_prime2[0]
    for G in G_prime2[1:]:
        y_ = G[1]
        if y_ < y_min:
            G_opt = G
    ##
    end_time = time.perf_counter()
    if runtime == True:
        text = f"Runtime (CPU: Ryzen 7 8700F; DDR5 RAM 16GB Dual): {round(end_time - start_time, 2)} sec" 
        print(text)
        print("-" *len(text))
    ##
    return G_opt

## Search Space and Initial Setup

In [3]:
# Generating Search Space
n = [4, 3]
d = [1, 3, 14, 1]
search_space = [
    [x, y, n, d]
    for x in range(1, 12 + 1)
    for y in range(1, 3 + 1)
]

### Size of Search Space

In [4]:
len(search_space)

36

## Computing the Optimized Solution & Runtime Benchmark

In [5]:
G_opt = optimizeCGS(search_space, runtime= True)
display(Latex(r"Optimal $(|A_p|, |A_s|, \mathbf{n}, \mathbf{d}) =$ " + f"{G_opt}"))
print("CGS-FO: " + f"{psi(G_opt)}")

Runtime (CPU: Ryzen 7 8700F; DDR5 RAM 16GB Dual): 1.1 sec
---------------------------------------------------------


<IPython.core.display.Latex object>

CGS-FO: 65


## CGS-FO of Other CGS Spaces

In [6]:
set([psi(G) for G in search_space])

{65, 82, 118, 230}